# F1 Trivia RAG — experimental run

End-to-end smoke test of the chatbot: ingest one real season of Ergast race results,
build the Gemini-embedded Chroma index, then ask the citation-aware query engine a
few trivia questions and inspect the answers + their source citations.

Scope is kept to a single season (2023) to keep the ingestion + embedding calls fast.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)  # so f1_trivia_rag.config picks up ./.env
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\thoma\f1-trivia-rag


## 1. Ingest

Pull every 2023 race result from the Ergast (jolpica) API and normalize into `RawDocument`s.

In [2]:
from f1_trivia_rag.ingestion.ergast import fetch_season_results

documents = fetch_season_results(2023)
print(f"Fetched {len(documents)} race-result documents")
print(documents[0].source_id)
print(documents[0].text)

Fetched 22 race-result documents
2023-1-result
Bahrain Grand Prix (2023), held at Bahrain International Circuit on 2023-03-05.
P1: Max Verstappen (Red Bull) - Finished
P2: Sergio Pérez (Red Bull) - Finished
P3: Fernando Alonso (Aston Martin) - Finished
P4: Carlos Sainz (Ferrari) - Finished
P5: Lewis Hamilton (Mercedes) - Finished
P6: Lance Stroll (Aston Martin) - Finished
P7: George Russell (Mercedes) - Finished
P8: Valtteri Bottas (Alfa Romeo) - Finished
P9: Pierre Gasly (Alpine F1 Team) - Finished
P10: Alexander Albon (Williams) - Finished
P11: Yuki Tsunoda (AlphaTauri) - Finished
P12: Logan Sargeant (Williams) - Lapped
P13: Kevin Magnussen (Haas F1 Team) - Lapped
P14: Nyck de Vries (AlphaTauri) - Lapped
P15: Nico Hülkenberg (Haas F1 Team) - Lapped
P16: Guanyu Zhou (Alfa Romeo) - Lapped
P17: Lando Norris (McLaren) - Lapped
P18: Esteban Ocon (Alpine F1 Team) - Retired
P19: Charles Leclerc (Ferrari) - Retired
P20: Oscar Piastri (McLaren) - Retired


## 2. Build the index

Embeds every document with Gemini (`models/gemini-embedding-001`) and persists to the
project's Chroma store at `storage/chroma` (same location `scripts/ingest.py` uses).

In [3]:
from f1_trivia_rag.rag.build_index import build_index

index = build_index(documents)
print("Index built and persisted to storage/chroma")

C:\Users\thoma\f1-trivia-rag\.venv\Lib\site-packages\llama_index\embeddings\gemini\base.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as gemini


Index built and persisted to storage/chroma


## 3. Load the query engine and chat

Same `load_query_engine()` the FastAPI `/chat` endpoint uses — citation-aware, top-5 retrieval.

In [4]:
from f1_trivia_rag.rag.query_engine import load_query_engine

engine = load_query_engine()

In [5]:
def ask(question: str) -> None:
    response = engine.query(question)
    print(f"Q: {question}")
    print(f"A: {response}\n")
    print("Citations:")
    for node in response.source_nodes:
        meta = node.metadata
        print(f"  - {meta.get('source')}:{meta.get('source_id')} ({meta.get('race_name')})")
    print("-" * 60)

In [6]:
ask("Who won the 2023 Monaco Grand Prix?")

Q: Who won the 2023 Monaco Grand Prix?
A: Max Verstappen (Red Bull) won the 2023 Monaco Grand Prix [1].

Citations:
  - ergast:2023-6-result (Monaco Grand Prix)
  - ergast:2023-5-result (Miami Grand Prix)
  - ergast:2023-7-result (Spanish Grand Prix)
  - ergast:2023-4-result (Azerbaijan Grand Prix)
  - ergast:2023-1-result (Bahrain Grand Prix)
------------------------------------------------------------


In [7]:
ask("Which constructor won the most races in the 2023 season?")

Q: Which constructor won the most races in the 2023 season?
A: Red Bull won the most races in the 2023 season, winning five races: the Austrian Grand Prix [1], the Miami Grand Prix [2], the Monaco Grand Prix [3], the Dutch Grand Prix [4], and the British Grand Prix [5].

Citations:
  - ergast:2023-9-result (Austrian Grand Prix)
  - ergast:2023-5-result (Miami Grand Prix)
  - ergast:2023-6-result (Monaco Grand Prix)
  - ergast:2023-13-result (Dutch Grand Prix)
  - ergast:2023-10-result (British Grand Prix)
------------------------------------------------------------


In [8]:
ask("Did any driver fail to finish the 2023 Australian Grand Prix, and why?")

Q: Did any driver fail to finish the 2023 Australian Grand Prix, and why?
A: Yes, several drivers failed to finish the 2023 Australian Grand Prix [1].

The drivers who retired were:
*   Pierre Gasly (Alpine F1 Team) [1]
*   Esteban Ocon (Alpine F1 Team) [1]
*   Nyck de Vries (AlphaTauri) [1]
*   Logan Sargeant (Williams) [1]
*   Kevin Magnussen (Haas F1 Team) [1]
*   George Russell (Mercedes) [1]
*   Alexander Albon (Williams) [1]
*   Charles Leclerc (Ferrari) [1]

Citations:
  - ergast:2023-3-result (Australian Grand Prix)
  - ergast:2023-1-result (Bahrain Grand Prix)
  - ergast:2023-2-result (Saudi Arabian Grand Prix)
  - ergast:2023-16-result (Japanese Grand Prix)
  - ergast:2023-6-result (Monaco Grand Prix)
------------------------------------------------------------
